# Inferencing — Ad-Click Classifier
**Platform:** Databricks Serverless  
**Input:** Raw impression data in the same schema as training CSV  
**Goal:** Apply the full FE pipeline, score with the registered champion model, write predictions to Delta, and run lightweight drift monitoring.

In [1]:
# Section 0a — pip installs (own cell)
%pip install -q seaborn scipy scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Section 0b — imports & constants
# %run ./utils/fe_helpers      # ← uncomment on Databricks
# %run ./utils/model_helpers   # ← uncomment on Databricks

import sys, os, json, math
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), 'utils'))
from fe_helpers import (
    assert_columns,
    add_cyclical_encoding, add_age_group,
    add_outlier_flag, apply_target_encoding, add_keyword_flags,
    plt, sns, pd, np, F,
)
from model_helpers import (
    plot_drift_report, log_figures_to_mlflow, log_metrics_to_mlflow,
)

from pyspark.sql import SparkSession
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window
from pyspark.ml import PipelineModel
from pyspark.ml.feature import VectorAssembler
from scipy.stats import ks_2samp
import mlflow
import mlflow.spark

# ── Constants ─────────────────────────────────────────────────────────────────
# ── Resolve absolute paths (Databricks Serverless + Asset Bundle) ─────────
# spark.read/write require absolute paths. dbutils gives the notebook's
# Workspace location at runtime; /Workspace prefix makes it unambiguous.
try:
    _nb_path = (
        dbutils.notebook.entry_point
        .getDbutils().notebook().getContext()
        .notebookPath().get()
    )
    _ws_dir = "/Workspace" + "/".join(_nb_path.split("/")[:-1])
except NameError:
    _ws_dir = "."   # local / non-Databricks fallback
INFERENCE_INPUT_PATH = f"{_ws_dir}/ad_10000records.csv"  # swap for new impression data path
FE_ARTIFACTS_PATH    = f"{_ws_dir}/fe_output/artifacts"
OUTPUT_PATH          = f"{_ws_dir}/inference_output"
MODEL_NAME           = "ad_click_classifier"
MODEL_STAGE          = "Production"               # or a version number e.g. "1"

PREDICTION_THRESHOLD = 0.5                        # binary classification cutoff
DRIFT_KS_ALPHA       = 0.05                       # p-value threshold for drift alert
LABEL_COL            = "label"
TARGET_COL           = "Clicked on Ad"
FEATURES_COL         = "features"
TIMESTAMP_COL        = "Timestamp"
RANDOM_SEED          = 42
TE_ALPHA             = 10.0

mlflow.set_experiment("ad_click_inferencing")

spark = SparkSession.builder.appName("ad_click_inference").getOrCreate()
print(f"Spark {spark.version} ready")


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
2026/04/29 20:24:06 INFO mlflow.tracking.fluent: Experiment with name 'ad_click_inferencing' does not exist. Creating a new experiment.
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/29 20:24:07 WARN Utils: Your hostname, Dudu, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/29 20:24:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 20:24:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/29 20:24:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting p

Spark 4.1.1 ready


## 1 · Load & Validate Inference Data
Load raw impressions and assert the expected schema before any transformation. A schema mismatch here surfaces data pipeline issues early — fail fast rather than propagate bad data.

In [3]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INFERENCE_INPUT_PATH)
)

REQUIRED_COLS = [
    "Daily Time Spent on Site", "Age", "Area Income",
    "Daily Internet Usage", "Ad Topic Line", "City",
    "Gender", "Country", TIMESTAMP_COL,
]
assert_columns(df_raw, REQUIRED_COLS)

n_input = df_raw.count()
print(f"Input rows: {n_input:,}")

# Null guard — log warning, do not silently drop
null_counts = {
    c: df_raw.filter(F.col(c).isNull()).count() for c in REQUIRED_COLS
}
nulls_found = {k: v for k, v in null_counts.items() if v > 0}
if nulls_found:
    print(f"WARNING: Unexpected nulls found: {nulls_found}")
else:
    print("Null check passed — no missing values.")

df_raw.limit(3).toPandas()

Input rows: 10,000
Null check passed — no missing values.


,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Ad Topic Line,City,Gender,Country,Timestamp,Clicked on Ad
0,62.26,32.0,69481.85,172.83,Decentralized real-time circuit,Lisafort,Male,Svalbard & Jan Mayen Islands,2016-06-09 21:43:05,0
1,41.73,31.0,61840.26,207.17,Optional full-range projection,West Angelabury,Male,Singapore,2016-01-16 17:56:05,0
2,44.40,30.0,57877.15,172.83,Total 5thgeneration standardization,Reyesfurt,Female,Guadeloupe,2016-06-29 10:50:45,0


## 2 · Apply Feature Engineering Pipeline
Every transformation is applied in the **exact same order** as `feature_engineering.ipynb` using the saved artifacts. No encoder is refit — all params come from training time.

In [4]:
# ── 2a: Temporal features ────────────────────────────────────────────────────
df = (
    df_raw
    .withColumn(TIMESTAMP_COL, F.col(TIMESTAMP_COL).cast(TimestampType()))
    .withColumn("hour",        F.hour(TIMESTAMP_COL))
    .withColumn("day_of_week", F.dayofweek(TIMESTAMP_COL))
    .withColumn("month",       F.month(TIMESTAMP_COL).cast("int"))
    .withColumn("is_weekend",  (F.dayofweek(TIMESTAMP_COL).isin(1, 7)).cast("int"))
)
df = add_cyclical_encoding(df, "hour",        period=24)
df = add_cyclical_encoding(df, "day_of_week", period=7)
df = df.drop(TIMESTAMP_COL, "hour", "day_of_week")
print("Temporal features applied.")

Temporal features applied.


In [5]:
# ── 2b: Numeric features ─────────────────────────────────────────────────────
df = df.withColumn(
    "usage_time_interaction",
    F.col("Daily Internet Usage") * F.col("Daily Time Spent on Site"),
)
df = df.withColumn("log_area_income", F.log1p(F.col("Area Income")))
df = add_age_group(df, col="Age")
df = df.withColumn(
    "income_quartile",
    F.ntile(4).over(Window.orderBy("Area Income")).cast("int"),
)

# Load saved IQR bounds (from training — never recompute on inference data)
with open(f"{FE_ARTIFACTS_PATH}/outlier_bounds.json") as f:
    outlier_bounds_map = json.load(f)

NUMERIC_FOR_OUTLIERS = [
    "Daily Time Spent on Site", "Age", "Area Income", "Daily Internet Usage"
]
for col in NUMERIC_FOR_OUTLIERS:
    df = add_outlier_flag(df, col, outlier_bounds_map[col])

print("Numeric features applied.")

FileNotFoundError: [Errno 2] No such file or directory: 'fe_output/artifacts/outlier_bounds.json'

In [ ]:
# ── 2c: Categorical encoding ─────────────────────────────────────────────────
# Gender — load fitted pipeline
gender_model = PipelineModel.load(f"{FE_ARTIFACTS_PATH}/gender_pipeline")
df = gender_model.transform(df).drop("Gender", "gender_idx")

# Country & City target encoding — load saved maps
with open(f"{FE_ARTIFACTS_PATH}/country_te_map.json") as f:
    country_map = json.load(f)
with open(f"{FE_ARTIFACTS_PATH}/city_te_map.json") as f:
    city_map = json.load(f)

# Compute global fallback from the maps themselves
global_rate = float(np.mean(list(country_map.values())))

df = apply_target_encoding(df, "Country", country_map, global_fallback=global_rate)
df = apply_target_encoding(df, "City",    city_map,    global_fallback=global_rate)

# City impression count — use training-time counts from saved map cardinality
# (approximation: log of rank in map as a frequency proxy)
city_rank_map = {k: math.log1p(i + 1) for i, k in enumerate(city_map.keys())}
df = apply_target_encoding(df, "City", city_rank_map,
                            global_fallback=math.log1p(len(city_map) / 2))
df = df.withColumnRenamed("city_te", "log_city_imp_count").drop("Country", "City")

print("Categorical encoding applied.")

In [ ]:
# ── 2d: Text features ────────────────────────────────────────────────────────
df = add_keyword_flags(df)

tfidf_model = PipelineModel.load(f"{FE_ARTIFACTS_PATH}/tfidf_pipeline")
df = tfidf_model.transform(df)
df = df.drop("Ad Topic Line", "tokens_raw", "tokens", "tf_features", "tfidf_features")

print("Text features applied.")

## 3 · Final Feature Assembly
Load the saved scale pipeline (VectorAssembler + MinMaxScaler) to produce `features_scaled`, then assemble the final `features` vector — identical to the assembly used at training time.

In [ ]:
# Scalar features through saved scaler
scale_model = PipelineModel.load(f"{FE_ARTIFACTS_PATH}/scale_pipeline")
df = scale_model.transform(df)

# Final assembly: features_scaled + gender_ohe + tfidf_selected → features
final_assembler = VectorAssembler(
    inputCols=["features_scaled", "gender_ohe", "tfidf_selected"],
    outputCol=FEATURES_COL,
    handleInvalid="keep",
)
df_features = final_assembler.transform(df)

print(f"Feature vector dimension: {len(df_features.select(FEATURES_COL).first()[0])}")
df_features.select(FEATURES_COL).limit(2).toPandas()

## 4 · Load & Apply Champion Model
Load the champion from the local artifact path (or from the MLflow Registry if running on a connected workspace). Score all impressions and extract per-row click probability.

In [ ]:
# Load from local artifact (fallback if registry is not reachable)
try:
    model_uri = f"models:/{MODEL_NAME}/{MODEL_STAGE}"
    champion = mlflow.spark.load_model(model_uri)
    model_source = f"MLflow Registry: {model_uri}"
except Exception:
    champion = PipelineModel.load(f"{FE_ARTIFACTS_PATH}/champion_model")
    model_source = f"Local: {FE_ARTIFACTS_PATH}/champion_model"

print(f"Model loaded from: {model_source}")

In [ ]:
# Score
raw_predictions = champion.transform(df_features)

# Extract scalar click probability from the probability vector
extract_prob = F.udf(lambda v: float(v[1]), "double")

predictions = (
    raw_predictions
    .withColumn("click_probability", extract_prob(F.col("probability")))
    .withColumn(
        "predicted_click",
        (F.col("click_probability") >= PREDICTION_THRESHOLD).cast("int"),
    )
    .withColumn("inference_timestamp", F.current_timestamp())
    .withColumn("model_source", F.lit(model_source))
)

n_predicted    = predictions.count()
predicted_ctr  = predictions.agg(F.mean("click_probability")).collect()[0][0]
pred_dist      = predictions.groupBy("predicted_click").count().toPandas()

print(f"Scored rows     : {n_predicted:,}")
print(f"Predicted CTR   : {predicted_ctr:.3f}")
print(pred_dist.to_string(index=False))

## 5 · Post-Process & Enrich Output
Select business-relevant columns alongside predictions and compute confidence distribution.

In [ ]:
# Re-join raw input columns with predictions for full business context
output_cols_raw = [
    "Daily Time Spent on Site", "Age", "Area Income",
    "Daily Internet Usage", "Ad Topic Line", "City",
    "Gender", "Country",
]
# Add monotonically increasing ID to enable the join
df_raw_id = df_raw.withColumn("_row_id", F.monotonically_increasing_id())
df_pred_id = predictions.withColumn("_row_id", F.monotonically_increasing_id())

output_df = (
    df_raw_id.select(["_row_id"] + output_cols_raw)
    .join(
        df_pred_id.select(
            "_row_id", "click_probability", "predicted_click",
            "inference_timestamp", "model_source"
        ),
        on="_row_id", how="inner"
    )
    .drop("_row_id")
)

output_df.limit(5).toPandas()

In [ ]:
# Probability distribution histogram
prob_pdf = predictions.select("click_probability").toPandas()

fig_prob, ax = plt.subplots(figsize=(9, 4))
ax.hist(prob_pdf["click_probability"], bins=50, color="steelblue", edgecolor="white")
ax.axvline(PREDICTION_THRESHOLD, color="#EE6644", lw=2, linestyle="--",
           label=f"Threshold = {PREDICTION_THRESHOLD}")
ax.set_xlabel("Click Probability")
ax.set_ylabel("Count")
ax.set_title("Predicted Click Probability Distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 6 · Write Predictions to Delta
Append predictions to the output Delta table. `mode=append` ensures historical predictions are preserved across multiple inference runs.

In [ ]:
output_df.write.format("delta").mode("append").save(f"{OUTPUT_PATH}/predictions")

# Verify written rows
written = spark.read.format("delta").load(f"{OUTPUT_PATH}/predictions").count()
print(f"Rows in predictions table (total cumulative): {written:,}")
spark.read.format("delta").load(f"{OUTPUT_PATH}/predictions").limit(3).toPandas()

## 7 · Data Drift Monitoring
Run a Kolmogorov-Smirnov test on every numeric feature comparing the inference batch against training reference statistics. A significant KS statistic (p < α) signals potential distribution shift that could degrade model performance.

In [ ]:
# Load training reference stats
with open(f"{FE_ARTIFACTS_PATH}/train_stats.json") as f:
    train_stats = json.load(f)

# Numeric columns to monitor (must exist in both train stats and inference data)
DRIFT_COLS = [
    "Daily Time Spent on Site", "Age", "Area Income", "Daily Internet Usage"
]
drift_rows = []

for col in DRIFT_COLS:
    if col not in train_stats:
        continue
    s = train_stats[col]
    # Reconstruct synthetic training distribution from saved stats (normal approximation)
    np.random.seed(RANDOM_SEED)
    train_sample = np.random.normal(s["mean"], max(s["std"], 1e-6), 1000)
    train_sample = np.clip(train_sample, s["min"], s["max"])

    inf_sample = df_raw.select(col).dropna().toPandas()[col].values

    ks_stat, p_val = ks_2samp(train_sample, inf_sample)
    drift_rows.append({
        "feature":    col,
        "ks_stat":    round(ks_stat, 4),
        "p_value":    round(p_val, 4),
        "drift_flag": bool(p_val < DRIFT_KS_ALPHA),
    })

drift_pdf = pd.DataFrame(drift_rows)
print(drift_pdf.to_string(index=False))

In [ ]:
fig_drift = plot_drift_report(drift_pdf, alpha=DRIFT_KS_ALPHA)
plt.show()

n_drift_alerts = drift_pdf["drift_flag"].sum()
if n_drift_alerts > 0:
    alerted = drift_pdf[drift_pdf["drift_flag"]]["feature"].tolist()
    print(f"\nDRIFT ALERT ({n_drift_alerts} feature(s)): {alerted}")
    print("Consider retraining if drift persists across multiple inference batches.")
else:
    print("\nNo drift detected.")

In [ ]:
# Write drift report to Delta
drift_sdf = spark.createDataFrame(drift_pdf).withColumn(
    "report_timestamp", F.current_timestamp()
)
drift_sdf.write.format("delta").mode("append").save(f"{OUTPUT_PATH}/drift_report")
print(f"Drift report written → {OUTPUT_PATH}/drift_report")

## 8 · MLflow Logging
Log all inference-run parameters, metrics, and artifacts for auditability.

In [ ]:
with mlflow.start_run(run_name="ad_click_inference_batch"):
    mlflow.log_params({
        "model_name":           MODEL_NAME,
        "model_stage":          MODEL_STAGE,
        "n_input_rows":         n_input,
        "prediction_threshold": PREDICTION_THRESHOLD,
        "drift_alpha":          DRIFT_KS_ALPHA,
        "input_path":           INFERENCE_INPUT_PATH,
    })
    mlflow.log_metrics({
        "n_predictions":    float(n_predicted),
        "predicted_ctr":    round(predicted_ctr, 4),
        "n_drift_alerts":   float(n_drift_alerts),
        "prob_mean":        round(float(prob_pdf["click_probability"].mean()), 4),
        "prob_std":         round(float(prob_pdf["click_probability"].std()),  4),
    })

    # Drift metrics
    for _, row in drift_pdf.iterrows():
        safe = row["feature"].replace(" ", "_")
        mlflow.log_metric(f"ks_{safe}",    row["ks_stat"])
        mlflow.log_metric(f"pval_{safe}",  row["p_value"])

    # Artifacts
    drift_csv = "/tmp/drift_report.csv"
    drift_pdf.to_csv(drift_csv, index=False)
    mlflow.log_artifact(drift_csv, "reports")

    log_figures_to_mlflow({
        "probability_distribution": fig_prob,
        "drift_report":             fig_drift,
    })

    print(f"MLflow run logged: {mlflow.active_run().info.run_id}")